## 0. Google Colab Setup

Mount Google Drive to access the project directory, and install dependencies. Run this cell first in every session to establish the working path.

### Mount Google Drive

Mount Google Drive to make the project directory available at `/content/drive/MyDrive/`. This must run before any path resolution or file access.

**Expected output:** A confirmation that Drive is already mounted or a prompt to authorize access.

In [ ]:
from google.colab import drive
# Mount to the standard base directory
drive.mount('/content/drive')

# Now you can define your project path and use it
project_path = '/content/drive/MyDrive/multimodal-causal-ablation'
import os
if os.path.exists(project_path):
    print(f'Successfully accessed: {project_path}')
else:
    print(f'Drive mounted, but folder not found: {project_path}')

!pip install -r /content/drive/MyDrive/multimodal-causal-ablation/requirements.txt

# Phase A — Dominant Modality Verification

Verify Audio as the dominant modality via aggregated DeepSHAP attribution, following the methodology locked in [ADR 0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Formally confirm which modality contributes the highest aggregated DeepSHAP attribution score for both the base and fine-tuned models. This is the prerequisite gate before any neuron-level probing or ablation work in Phases B–D.

## 1. Environment & Imports

Set up the deterministic seed (`seed=0`, matching upstream checkpoint convention) and import all required libraries. The seed utility from `src/utils.py` pins `torch`, `numpy`, `random`, and `cudnn` for full reproducibility across ephemeral Colab runtimes.

**Expected output:** Confirmation of project path, checkpoints directory, and results directory.

In [ ]:
import sys
import os
import pickle

import numpy as np
import pandas as pd

# Add project src to path for utility imports
sys.path.insert(0, os.path.join(project_path, 'src'))
from utils import set_deterministic_seed

# Pin all randomness sources (seed=0 matches upstream checkpoint convention)
set_deterministic_seed(seed=0)

# Define paths
checkpoints_dir = os.path.join(project_path, 'checkpoints')
results_dir = os.path.join(project_path, 'results')
os.makedirs(results_dir, exist_ok=True)

print(f'Project path:  {project_path}')
print(f'Checkpoints:   {checkpoints_dir}')
print(f'Results:       {results_dir}')

## 2. Load Pre-computed DeepSHAP Attributions

Load the pre-computed DeepSHAP attribution pickles for both the base and fine-tuned models. Each pickle contains a dict with:
- `'SHAP_value'`: list of 6 numpy arrays (one per emotion class), each shaped `(n_samples, 1152)`
- `'test_feature'`: corresponding test input features

The 1152-dimensional feature vector is a concatenation of three modality representations:
- **Text** (ALBERT): dimensions 0–1023 (1024-d)
- **Video** (visual): dimensions 1024–1087 (64-d)
- **Audio** (acoustic): dimensions 1088–1151 (64-d)

**Expected output:** Structure summary showing keys, number of classes, and per-class array shapes for both models.

In [ ]:
# Load pre-computed DeepSHAP attributions for both models
with open(os.path.join(checkpoints_dir, 'base_shap.pkl'), 'rb') as f:
    base_shap_data = pickle.load(f)

with open(os.path.join(checkpoints_dir, 'finetuned_shap.pkl'), 'rb') as f:
    finetuned_shap_data = pickle.load(f)

# Inspect structure
print('=== Base Model SHAP ===')
print(f'Keys: {list(base_shap_data.keys())}')
print(f'Number of classes: {len(base_shap_data["SHAP_value"])}')
for i, sv in enumerate(base_shap_data['SHAP_value']):
    print(f'  Class {i}: shape = {sv.shape}')

print()
print('=== Fine-tuned Model SHAP ===')
print(f'Keys: {list(finetuned_shap_data.keys())}')
print(f'Number of classes: {len(finetuned_shap_data["SHAP_value"])}')
for i, sv in enumerate(finetuned_shap_data['SHAP_value']):
    print(f'  Class {i}: shape = {sv.shape}')

## 3. Compute Aggregated SHAP Attribution per Modality

Raw 1152-d SHAP values create a **dimensionality illusion**: Text has 16× more features than Audio or Video, so naive per-feature comparisons inflate Text's apparent contribution.

Per ADR 0001, I resolve this by computing **aggregated** SHAP attribution:
1. For each emotion class, take the absolute value of all SHAP values.
2. Sum |SHAP| within each modality for every sample — this collapses each modality's contribution to a single scalar per sample.
3. Average across samples to get one attribution score per modality per class.
4. Average across classes to get the overall modality attribution.

This makes the comparison fair regardless of how many raw features each modality contributes.

**Expected output:** Per-class attribution percentages for Text, Video, and Audio in both models.

In [ ]:
# Modality feature ranges in the 1152-d concatenated feature vector
MODALITY_RANGES = {
    'Text':  (0, 1024),     # ALBERT embeddings (1024-d)
    'Video': (1024, 1088),  # Visual features (64-d)
    'Audio': (1088, 1152),  # Acoustic features (64-d)
}

EMOTION_CLASSES = [
    'anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise'
]


def compute_aggregated_shap(shap_data, model_name):
    """Compute aggregated SHAP attribution per modality.

    For each class, sum |SHAP values| within each modality per sample,
    then average across samples. This is the aggregation method from
    ADR 0001 to resolve the dimensionality illusion between Text
    (1024-d) and Audio (64-d).
    """
    shap_values = shap_data['SHAP_value']
    n_classes = len(shap_values)

    rows = []
    for class_idx in range(n_classes):
        sv = shap_values[class_idx]  # (n_samples, 1152)

        row = {'model': model_name, 'class': EMOTION_CLASSES[class_idx]}
        for mod_name, (start, end) in MODALITY_RANGES.items():
            # Sum |SHAP| within modality per sample, then mean across samples
            per_sample = np.sum(np.abs(sv[:, start:end]), axis=1)
            row[f'{mod_name}_attribution'] = np.mean(per_sample)
            row[f'{mod_name}_std'] = np.std(per_sample)

        # Percentages for readability
        total = sum(row[f'{m}_attribution'] for m in MODALITY_RANGES)
        for mod_name in MODALITY_RANGES:
            row[f'{mod_name}_pct'] = (
                row[f'{mod_name}_attribution'] / total * 100
            )

        rows.append(row)

    return pd.DataFrame(rows)


# Compute for both models
base_attr_df = compute_aggregated_shap(base_shap_data, 'base')
finetuned_attr_df = compute_aggregated_shap(finetuned_shap_data, 'finetuned')

# Combine results
attribution_df = pd.concat(
    [base_attr_df, finetuned_attr_df], ignore_index=True
)

# Display per-class results
print('=== Per-Class Aggregated SHAP Attribution (%) ===')
print()
display_cols = ['model', 'class', 'Text_pct', 'Video_pct', 'Audio_pct']
print(
    attribution_df[display_cols].to_string(
        index=False, float_format='%.2f'
    )
)

### Apply Dominant Modality Decision Rule

Apply the ADR 0001 decision rule to the aggregated SHAP attributions to formally select the dominant modality. If the gap between the top two modalities is within 5%, the Audio branch (64-d) is chosen for its superior neuron-to-class ratio. Otherwise, the clear winner is selected.

**Expected output:** A printed verdict for both base and fine-tuned models, and two saved CSV files: `phase_a_dominant_modality_verdict.csv` and `phase_a_shap_attribution_by_class.csv`.

In [ ]:
def compute_verdict(attr_df, model_name):
    """Apply ADR 0001 dominant modality decision rule.

    Returns a dict with the verdict and supporting evidence.
    """
    model_df = attr_df[attr_df['model'] == model_name]

    # Overall attribution: mean across classes
    summary = {}
    for mod_name in MODALITY_RANGES:
        summary[mod_name] = model_df[f'{mod_name}_attribution'].mean()

    total = sum(summary.values())
    pct = {k: v / total * 100 for k, v in summary.items()}

    # Rank by attribution
    ranked = sorted(pct.items(), key=lambda x: x[1], reverse=True)

    sep = '=' * 55
    print(sep)
    print(f'  {model_name.upper()} MODEL — Aggregated SHAP Attribution')
    print(sep)
    for mod, p in ranked:
        print(f'  {mod:8s}: {p:6.2f}%  (raw mean: {summary[mod]:.6f})')

    top_mod, top_pct = ranked[0]
    second_mod, second_pct = ranked[1]
    gap = top_pct - second_pct

    # ADR 0001 decision rule
    if gap <= 5.0:
        dominant = 'Audio'
        reason = (
            f'Top two modalities ({top_mod}: {top_pct:.2f}%, '
            f'{second_mod}: {second_pct:.2f}%) are within 5% parity '
            f'(gap = {gap:.2f}%). Per ADR 0001, targeting Audio (64-d) '
            f'for superior neuron-to-class ratio.'
        )
    else:
        dominant = top_mod
        reason = (
            f'{top_mod} leads with {top_pct:.2f}% vs '
            f'{second_mod} at {second_pct:.2f}% '
            f'(gap = {gap:.2f}% > 5% threshold).'
        )

    print()
    print(f'  VERDICT: Dominant Modality = {dominant}')
    print(f'  Reason:  {reason}')

    return {
        'model': model_name,
        'dominant_modality': dominant,
        'Text_pct': round(pct['Text'], 4),
        'Video_pct': round(pct['Video'], 4),
        'Audio_pct': round(pct['Audio'], 4),
        'top_modality': top_mod,
        'second_modality': second_mod,
        'gap_pct': round(gap, 4),
        'parity_rule_applied': gap <= 5.0,
        'reason': reason,
    }


# Apply verdict to both models
base_verdict = compute_verdict(attribution_df, 'base')
print()
finetuned_verdict = compute_verdict(attribution_df, 'finetuned')

# --- Save results ---
verdict_df = pd.DataFrame([base_verdict, finetuned_verdict])
verdict_path = os.path.join(
    results_dir, 'phase_a_dominant_modality_verdict.csv'
)
verdict_df.to_csv(verdict_path, index=False)

detail_path = os.path.join(
    results_dir, 'phase_a_shap_attribution_by_class.csv'
)
attribution_df.to_csv(detail_path, index=False)

print()
print(f'Results saved:')
print(f'  Verdict:  {verdict_path}')
print(f'  Details:  {detail_path}')

# --- Summary ---
sep = '=' * 55
print()
print(sep)
print('  PHASE A SUMMARY')
print(sep)
bm = base_verdict['dominant_modality']
fm = finetuned_verdict['dominant_modality']
print(f'  Base model dominant modality:       {bm}')
print(f'  Fine-tuned model dominant modality:  {fm}')

if base_verdict['dominant_modality'] == 'Audio':
    print()
    print('  ✓ Audio confirmed as dominant modality.')
    print('    Proceed to Phase B: Probe Signal Validation '
          'on Audio FFN activations.')
else:
    dm = base_verdict['dominant_modality']
    print()
    print(f'  ✗ Audio NOT confirmed. Dominant modality = {dm}.')
    print('    Review methodology — experiment targets '
          'the dominant modality.')

## 3.5 Full-Dataset Activation Extraction (P1 Fix)

The SHAP pickles only cached 144 test samples. Per the experiment protocol (Day 1-2), activation statistics and ablation evaluations should use the full RML dataset (train+val+test combined = 723 samples) since we are not training anything — just observing neuron behavior and computing selectivity statistics.

This cell loads the complete RML dataset, runs `SHAP_feature()` through both models, and caches the 1152-d representations. All downstream phases (B, C, D) will use these larger tensors.

**Expected output:** Confirmation of 723-sample feature tensors for both models, saved to `checkpoints/activations/`.

In [ ]:
import os
import sys
import pickle
import numpy as np
import torch
from torch.utils.data import DataLoader

# Ensure src and model code are importable
sys.path.insert(0, os.path.join(project_path, 'Model/Dig-Data_Model-Main'))
sys.path.insert(0, os.path.join(project_path, 'src'))

from utils import set_deterministic_seed
set_deterministic_seed(seed=0)

from src.datasets import IEMOCAP, collate_fn
from src.models.e2e import MME2E
from transformers import AlbertTokenizer

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Paths ──
data_dir = os.path.join(project_path, 'Model/Dig-Data_Model-Main/data')
main_folder = os.path.join(data_dir, 'RML_RAW_PROCESSED_Face')
split_dir = os.path.join(
    data_dir, 'data_split', 'all_single_label_six_category', 'with_valid'
)
activations_dir = os.path.join(project_path, 'checkpoints', 'activations')
os.makedirs(activations_dir, exist_ok=True)

# ── Load ALL split utterance IDs ──
train_ids = open(os.path.join(
    split_dir, 'Final_train_split_six_categories_RML.txt'
)).read().splitlines()
valid_ids = open(os.path.join(
    split_dir, 'Final_valid_split_six_categories_RML.txt'
)).read().splitlines()
test_ids = open(os.path.join(
    split_dir, 'Final_test_split_six_categories_RML.txt'
)).read().splitlines()
full_uttr_ids = train_ids + valid_ids + test_ids

print(f"Train: {len(train_ids)}, Val: {len(valid_ids)}, "
      f"Test: {len(test_ids)}, Total: {len(full_uttr_ids)}")

# ── Load metadata ──
with open(os.path.join(main_folder, 'meta.pkl'), 'rb') as f:
    meta = pickle.load(f)

emoDict = {'ang': 0, 'dis': 1, 'fea': 2, 'hap': 3, 'sad': 4, 'sur': 5}
# Handle missing transcripts (nan) gracefully
texts = [meta[uid]['text'] if isinstance(meta[uid]['text'], str) else "" for uid in full_uttr_ids]
labels_onehot = [np.eye(6)[emoDict[meta[uid]['label']]] for uid in full_uttr_ids]

# ── Build Dataset and DataLoader ──
full_dataset = IEMOCAP(
    main_folder=main_folder,
    utterance_ids=full_uttr_ids,
    texts=texts,
    labels=labels_onehot,
    label_annotations=list(emoDict.keys()),
    img_interval=500
)

full_loader = DataLoader(
    full_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print(f"Dataset size: {len(full_dataset)}")
print(f"DataLoader batches: {len(full_loader)}")

### 3.5.1 Forward Pass: Extract Full-Dataset SHAP Features

Run `SHAP_feature()` through both the base and fine-tuned models for every sample in the full RML dataset. This produces two `(723, 1152)` tensors — the exact pre-classification representations that feed into `t_out`, `v_out`, `a_out`.

**Expected output:** Per-model progress bars and final tensor shapes. Estimated runtime: ~10-30 minutes per model depending on GPU.

In [ ]:
from transformers import AlbertTokenizer
import os
import glob
import torch
import numpy as np

# ── Load tokenizer ──
tokenizer = AlbertTokenizer.from_pretrained('albert-large-v2')

# ── Model args (must match checkpoint training config) ──
model_args = {
    'num_emotions': 6,
    'modalities': 'tav',
    'feature_dim': 256,
    'trans_nlayers': 4,
    'trans_nheads': 4,
    'trans_dim': 64,
    'text_model_size': 'large',
    'text_max_len': 100,
}

def extract_full_features(model_path, model_name, cache_dir_name):
    """Run SHAP_feature() over the full RML dataset with incremental batch saving."""
    print(f"\n{'='*55}")
    print(f"  Extracting features: {model_name}")
    print(f"{'='*55}")

    # Set up temporary directory for caching batches
    cache_dir = os.path.join(activations_dir, cache_dir_name)
    os.makedirs(cache_dir, exist_ok=True)
    expected_batches = len(full_loader)
    
    # Check how many batches we've already saved
    cached_batch_files = glob.glob(os.path.join(cache_dir, "batch_features_*.npy"))
    if len(cached_batch_files) == expected_batches:
        print("  All batches found in cache! Skipping forward pass and loading from disk.")
    else:
        print(f"  Found {len(cached_batch_files)}/{expected_batches} cached batches. Resuming extraction...")
        model = MME2E(args=model_args, device=device).to(device)
        model.load_state_dict(
            torch.load(model_path, map_location=device), strict=False
        )
        model.eval()

    all_features = []
    all_labels = []
    processed = 0

    for batch_idx, batch in enumerate(full_loader):
        feature_file = os.path.join(cache_dir, f"batch_features_{batch_idx}.npy")
        label_file = os.path.join(cache_dir, f"batch_labels_{batch_idx}.npy")

        uttr_ids_batch, imgs, img_lens, specs, spec_lens, text_batch, Y = batch
        
        # If we already saved this batch, just load it
        if os.path.exists(feature_file) and os.path.exists(label_file):
            features_np = np.load(feature_file)
            labels_np = np.load(label_file)
        else:
            # Otherwise, run the model on this batch
            with torch.no_grad():
                text_inputs = tokenizer(
                    list(text_batch),
                    return_tensors='pt',
                    max_length=model_args['text_max_len'],
                    padding='max_length',
                    truncation=True
                ).to(device)

                specs = specs.to(device)

                features = model.SHAP_feature(
                    imgs, img_lens, specs, spec_lens, text_inputs
                )
                
                features_np = features.cpu().numpy()
                labels_np = Y.cpu().numpy() if hasattr(Y, 'cpu') else np.array(Y)
                
                # Immediately save this batch to Google Drive
                np.save(feature_file, features_np)
                np.save(label_file, labels_np)

        all_features.append(features_np)
        all_labels.append(labels_np)

        processed += len(uttr_ids_batch)
        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(full_loader):
            print(f"  Batch {batch_idx+1}/{len(full_loader)} "
                  f"({processed}/{len(full_dataset)} samples)")

    # Combine all batches into one large array
    all_features = np.concatenate(all_features, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Convert one-hot to integer labels
    int_labels = np.argmax(all_labels, axis=1)

    print(f"  Features shape: {all_features.shape}")
    print(f"  Labels shape:   {int_labels.shape}")

    # Free GPU memory
    if 'model' in locals():
        del model
        torch.cuda.empty_cache()

    return all_features, int_labels


# ── Extract for both models ──
base_full_1152, labels_full = extract_full_features(
    os.path.join(project_path, 'checkpoints', 'base_model.pt'),
    'Base Model',
    'tmp_base_features'
)

ft_full_1152, labels_full_ft = extract_full_features(
    os.path.join(project_path, 'checkpoints', 'finetuned_model.pt'),
    'Fine-tuned Model',
    'tmp_ft_features'
)

# Verify label ordering matches
assert np.array_equal(labels_full, labels_full_ft), \
    "Label ordering mismatch between models!"
print("\n✓ Label ordering verified: both models processed identical samples.")

### 3.5.2 Cache Full-Dataset Activations

Save the extracted features and labels to `checkpoints/activations/` for resume-gate loading in future sessions. Then overwrite the in-memory variables used by Phases B, C, and D so downstream cells automatically use the full dataset.

**Expected output:** Saved file paths and updated variable shapes.

In [ ]:
# ── Save to disk ──
np.save(os.path.join(activations_dir, 'base_full_1152.npy'), base_full_1152)
np.save(os.path.join(activations_dir, 'ft_full_1152.npy'), ft_full_1152)
np.save(os.path.join(activations_dir, 'labels_full.npy'), labels_full)

print("Saved to checkpoints/activations/:")
print(f"  base_full_1152.npy  -> {base_full_1152.shape}")
print(f"  ft_full_1152.npy    -> {ft_full_1152.shape}")
print(f"  labels_full.npy     -> {labels_full.shape}")

# ── Overwrite in-memory variables for downstream Phases ──
# These variable names are what Phase B/C/D cells expect
base_acts_tier1 = base_full_1152[:, 0:1024]
ft_acts_tier1   = ft_full_1152[:, 0:1024]
labels_tier1    = labels_full
base_1152       = base_full_1152
ft_1152         = ft_full_1152

print(f"\nIn-memory variables updated for downstream phases:")
print(f"  base_acts_tier1: {base_acts_tier1.shape}")
print(f"  ft_acts_tier1:   {ft_acts_tier1.shape}")
print(f"  labels_tier1:    {labels_tier1.shape}")
print(f"  base_1152:       {base_1152.shape}")
print(f"  ft_1152:         {ft_1152.shape}")

# ── Class distribution check ──
print(f"\nPer-class sample counts (full dataset):")
for c, name in enumerate(EMOTION_CLASSES):
    count = np.sum(labels_full == c)
    print(f"  {name:10s}: {count} samples ({count/len(labels_full)*100:.1f}%)")

# Phase B — Probe Signal Validation

Validate the target modality's signal via L1-logistic regression, following the methodology locked in [ADR         
0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Ensure the chosen modality branch (Text, 1024-d) retains sufficient class-discriminative information     
before we perform causal ablations. If the signal is too weak, we trigger a layer fallback.

## 4. Resume Gate: Load Cached Activations
Loads cached 1024-d Text branch activations to bypass expensive PyTorch forward passes.

**Expected output:** Confirmation of loaded tensor shapes.

In [ ]:
from utils import set_deterministic_seed
set_deterministic_seed(seed=0)
import os                                                                                                          
import pickle                                                                                                      
import numpy as np                                                                                                 
import torch                                                                                                       

# Use absolute paths for Colab                                                                                     
checkpoints_dir = os.path.join(project_path, 'checkpoints')                                                        
activations_dir = os.path.join(checkpoints_dir, 'activations')                                                     
data_dir = os.path.join(project_path, 'Model/Dig-Data_Model-Main/data')                                            
                                                                                                                    
print("Loading SHAP data to extract Tier 2 representations...")                                                    
with open(os.path.join(checkpoints_dir, 'base_shap.pkl'), 'rb') as f:                                              
    base_shap_data = pickle.load(f)                                                                                
                                                                                                                    
with open(os.path.join(checkpoints_dir, 'finetuned_shap.pkl'), 'rb') as f:                                         
    ft_shap_data = pickle.load(f)                                                                                  
                                                                                                                    
# 1. Extract the first 1024 dimensions (ALBERT CLS outputs)                                                        
base_test_feats = base_shap_data['test_feature']                                                                   
ft_test_feats = ft_shap_data['test_feature']                                                                       
                                                                                                                    
if torch.is_tensor(base_test_feats): base_test_feats = base_test_feats.cpu().numpy()                               
if torch.is_tensor(ft_test_feats): ft_test_feats = ft_test_feats.cpu().numpy()                                     
                                                                                                                    
base_acts_tier2 = base_test_feats[:, 0:1024]                                                                       
ft_acts_tier2 = ft_test_feats[:, 0:1024]                                                                           
                                                                                                                    
# Cache them specifically as tier 2
np.save(os.path.join(activations_dir, 'base_acts_tier2.npy'), base_acts_tier2)
np.save(os.path.join(activations_dir, 'ft_acts_tier2.npy'), ft_acts_tier2)

# 2. Extract True Labels directly from RML metadata
emoDict = {'ang': 0, 'dis': 1, 'fea': 2, 'hap': 3, 'sad': 4, 'sur': 5}
split_file = os.path.join(data_dir, 'data_split', 'all_single_label_six_category', 'with_valid', 'Final_test_split_six_categories_RML.txt')
meta_file = os.path.join(data_dir, 'RML_RAW_PROCESSED_Face', 'meta.pkl')

with open(split_file, 'r') as f:
    uttr_ids = f.read().splitlines()

with open(meta_file, 'rb') as f:
    meta = pickle.load(f)

# Map utterance ID -> string label -> integer label
labels_tier2 = np.array([emoDict[meta[uid]['label']] for uid in uttr_ids])

print(f"Successfully extracted Tier 2 (Encoder CLS) 1024-d representations and labels.")
print(f"  Base Tier 2 shape: {base_acts_tier2.shape}")
print(f"  FT Tier 2 shape:   {ft_acts_tier2.shape}")
print(f"  Labels shape:      {labels_tier2.shape}")

## 5. L1-Logistic Probe Validation (Tier 1)

*Note on Data Integrity:* Initial tests on a pre-computed `(288, 512)` cache showed zero signal. A forensic audit revealed this cache belonged to a different dataset (likely IEMOCAP video features). Furthermore, a review of `src/models/e2e.py` confirms that the Text branch lacks an intermediate Feed-Forward Network. The ALBERT Encoder CLS token (1024-d) routes directly to the classifier. Therefore, this 1024-d vector represents our true **Tier 1** representation.

Below, we extract the correct 1024-d text features and the corresponding 144 RML true labels to perform the Tier 1 validation.

**Expected output:** Per-class AUC scores, Mean AUC, and a PASS/WARNING fallback decision for both the base and fine-tuned models.

In [ ]:
import os
import numpy as np

checkpoints_dir = os.path.join(project_path, 'checkpoints')
activations_dir = os.path.join(checkpoints_dir, 'activations')

# Use full-dataset activations (720 samples) from 3.5.2 cache
if 'base_acts_tier1' not in globals() or len(base_acts_tier1) != 720:
    print("Loading 720-sample full dataset activations from disk cache...")
    base_acts_tier1 = np.load(os.path.join(activations_dir, 'base_full_1152.npy'))[:, 0:1024]
    ft_acts_tier1 = np.load(os.path.join(activations_dir, 'ft_full_1152.npy'))[:, 0:1024]

# Save pristine copies of full-dataset Tier 1 representations
np.save(os.path.join(activations_dir, 'base_acts_tier1.npy'), base_acts_tier1)
np.save(os.path.join(activations_dir, 'ft_acts_tier1.npy'), ft_acts_tier1)

print(f"Base Tier 1 shape: {base_acts_tier1.shape}")
print(f"FT Tier 1 shape:   {ft_acts_tier1.shape}")

### 5.1 Aligning Target Labels
The previously cached labels were tied to the corrupted 288-sample dataset. Here, we cleanly extract the true 144 target labels for the RML dataset by matching the test split utterance IDs directly against the dataset's `meta.pkl`.

In [ ]:
import os
import numpy as np

activations_dir = os.path.join(project_path, 'checkpoints', 'activations')

# Use full-dataset labels (720 samples) from 3.5.2 cache
if 'labels_tier1' not in globals() or len(labels_tier1) != 720:
    print("Loading 720-sample full dataset labels from disk cache...")
    labels_tier1 = np.load(os.path.join(activations_dir, 'labels_full.npy'))

print(f"Labels shape: {labels_tier1.shape}")

### 5.2 Out-of-Fold Evaluation
We define and run our out-of-fold `StratifiedKFold` logistic regression pipeline on the clean Tier 1 representations. If the Mean AUC exceeds 0.65 and at least 4 out of 6 classes exceed 0.55, the signal validation formally passes. Validation results are then saved to a CSV artifact.

**Expected output:** Per-class AUC scores, Mean AUC, PASS/WARNING fallback decisions, and CSV artifact paths.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from utils import set_deterministic_seed

EMOTION_CLASSES = [
    'anger',
    'disgust',
    'fear',
    'happiness',
    'sadness',
    'surprise',
]


def validate_probe_signal(acts, labels, model_name):
  set_deterministic_seed(seed=0)
  print(f'=== {model_name.upper()} MODEL PROBE VALIDATION (Tier 1) ===')
  n_classes = len(np.unique(labels))
  cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

  aucs = []
  results = []
  fitted_probes = {}

  for c in range(n_classes):
    y_binary = (labels == c).astype(int)

    # Scikit-learn Pipeline prevents pre-processing data leakage across CV folds (Issue B.3)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        (
            'probe',
            LogisticRegression(
                penalty='l1',
                solver='liblinear',
                class_weight='balanced',
                random_state=0,
                max_iter=1000,
            ),
        ),
    ])

    # Out-of-fold probability estimates for honest AUC evaluation
    probs = cross_val_predict(
        pipe, acts, y_binary, cv=cv, method='predict_proba'
    )[:, 1]
    auc = roc_auc_score(y_binary, probs)
    aucs.append(auc)
    results.append(
        {'model': model_name, 'class': EMOTION_CLASSES[c], 'auc': round(auc, 4)}
    )

    # Fit persistent probe on full data to retain L1 coefficients (Issue B.4)
    pipe.fit(acts, y_binary)
    fitted_probes[c] = pipe.named_steps['probe']

    print(f'  Class {EMOTION_CLASSES[c]:<10} AUC: {auc:.4f}')

  mean_auc = np.mean(aucs)
  poor_classes = sum(1 for a in aucs if a < 0.55)
  print('-' * 45)
  print(f'  Mean AUC: {mean_auc:.4f} (Threshold: >= 0.65)')
  print(f'  Classes < 0.55 AUC: {poor_classes} (Threshold: <= 2)')

  # Save CSV artifact
  df = pd.DataFrame(results)
  out_path = os.path.join(results_dir, f'phase_b_{model_name}_tier1_aucs.csv')
  df.to_csv(out_path, index=False)
  print(f'  [SAVED] {out_path}')

  return fitted_probes, (mean_auc >= 0.65 and poor_classes <= 2)


print('Validating True Tier 1 Signal...\n')
base_probes, base_valid = validate_probe_signal(
    base_acts_tier1, labels_tier1, 'base'
)
print()
ft_probes, ft_valid = validate_probe_signal(
    ft_acts_tier1, labels_tier1, 'finetuned'
)

# Phase C — Causal Ablation

Compute class-selectivity ratios and perform mean-ablation sweeps, following the methodology locked in [ADR 0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Identify the most causally active neurons for each emotion class in the 1024-d Text Tier 1 representation, and evaluate the classification accuracy drop when these specific neurons are ablated. A feature set is causally class-selective if the target class accuracy drop is $\ge 2.5\times$ the mean absolute non-target class drop.

## 6. Compute Class-Selectivity Ratios

To find the top $k$ neurons to ablate, we compute the selectivity ratio $\frac{\mu(n|c)}{\mu(n|\neg c)}$ for every neuron $n$ in the 1024-d representation across all emotion classes.

**Expected output:** A list of the top 5 highly selective neuron indices for each class in both the base and fine-tuned models.

In [ ]:
import numpy as np
from utils import set_deterministic_seed

EMOTION_CLASSES = [
    'anger',
    'disgust',
    'fear',
    'happiness',
    'sadness',
    'surprise',
]
K_VALUES = [1, 3, 5, 10]

set_deterministic_seed(seed=0)
print('=== Phase C: Ranking Neurons by Absolute L1 Probe Weights ===')


def rank_top_neurons_by_l1_weights(fitted_probes):
  """Ranks neurons by absolute L1 logistic regression probe weight magnitude (|coef_|).

  This directly identifies the features the linear classifier relies on for
  detecting each emotion (resolving Issue B.1 & protocol Day 3/11-12
  requirements).
  """
  top_neurons = {}
  top_weights = {}

  for c, probe in fitted_probes.items():
    # probe.coef_ has shape (1, n_features) for binary classification
    coefs = np.abs(probe.coef_[0])

    # Rank indices in descending order of absolute weight magnitude
    ranked_indices = np.argsort(coefs)[::-1]
    top_neurons[c] = ranked_indices
    top_weights[c] = coefs[ranked_indices]

  return top_neurons, top_weights


# Compute rankings using fitted L1 probes from Phase B
base_top_neurons, base_top_weights = rank_top_neurons_by_l1_weights(
    base_probes
)
ft_top_neurons, ft_top_weights = rank_top_neurons_by_l1_weights(ft_probes)

print('\nBase Model — Top 5 Neurons per Class (L1 Probe Weight Ranking):')
for c, idxs in base_top_neurons.items():
  weights_str = ', '.join([f'{w:.4f}' for w in base_top_weights[c][:5]])
  print(
      f'  {EMOTION_CLASSES[c]:<10}: Indices={idxs[:5]} | Weights=[{weights_str}]'
  )

print('\nFine-Tuned Model — Top 5 Neurons per Class (L1 Probe Weight Ranking):')
for c, idxs in ft_top_neurons.items():
  weights_str = ', '.join([f'{w:.4f}' for w in ft_top_weights[c][:5]])
  print(
      f'  {EMOTION_CLASSES[c]:<10}: Indices={idxs[:5]} | Weights=[{weights_str}]'
  )

weight_rows = []
for c in range(6):
  class_name = EMOTION_CLASSES[c]
  for rank in range(5):
    b_idx = base_top_neurons[c][rank]
    b_w = base_top_weights[c][rank]
    ft_idx = ft_top_neurons[c][rank]
    ft_w = ft_top_weights[c][rank]

    weight_rows.append({
        'class': class_name,
        'rank': rank + 1,
        'base_neuron_idx': b_idx,
        'base_l1_weight': round(b_w, 4),
        'ft_neuron_idx': ft_idx,
        'ft_l1_weight': round(ft_w, 4),
    })

df_weights = pd.DataFrame(weight_rows)
out_weights_path = os.path.join(results_dir, 'phase_b_top5_probe_weights.csv')
df_weights.to_csv(out_weights_path, index=False)
print(f'\n[SAVED] Day 5 Probe Weight Table: {out_weights_path}')

## 7. Mean-Ablation Proxy Inference & Model Loading

Because causal ablation must be measured via accuracy drops, we must evaluate the actual PyTorch `MME2E` models. However, instead of reloading the raw dataset (images, audio, text) and running the heavy encoders, we can leverage the exact 1152-d `test_feature` representations we already cached in Phase A. 

The `MME2E` architecture allows us to cleanly split this 1152-d vector back into `Text (1024-d)`, `Video (64-d)`, and `Audio (64-d)`, and pass them directly into the pre-trained classification heads (`t_out`, `v_out`, `a_out`, and `weighted_fusion`). This "proxy inference" mathematically perfectly simulates the forward pass while allowing us to seamlessly clamp the Text neurons.

**Expected output:** Loading of the base and fine-tuned `MME2E` PyTorch checkpoints, and definition of the `ablate_and_evaluate` proxy inference function.

In [ ]:
from utils import set_deterministic_seed
set_deterministic_seed(seed=0)
import sys
import os
import torch
import torch.nn as nn
import numpy as np

# Ensure MME2E is in path
sys.path.insert(0, os.path.join(project_path, 'Model/Dig-Data_Model-Main'))
from src.models.e2e import MME2E

# Setup mocked args for model instantiation (matches Phase A setup)
args = {
    'num_emotions': 6,
    'modalities': 'tav',
    'feature_dim': 256,
    'trans_nlayers': 4,
    'trans_nheads': 4,
    'trans_dim': 64,
    'text_model_size': 'large',
}
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print("Loading PyTorch MME2E models for ablation...")
base_model = MME2E(args=args, device=device).to(device)
base_model.load_state_dict(torch.load(os.path.join(checkpoints_dir, 'base_model.pt'), map_location=device), strict=False)
base_model.eval()

ft_model = MME2E(args=args, device=device).to(device)
ft_model.load_state_dict(torch.load(os.path.join(checkpoints_dir, 'finetuned_model.pt'), map_location=device), strict=False)
ft_model.eval()
print("Models loaded successfully.")

# Load full 1152-d representations (720 samples) from 3.5.2 cache
if 'base_1152' not in globals() or len(base_1152) != 720:
    print("Loading 720-sample 1152-d representations from disk cache...")
    base_1152 = np.load(os.path.join(activations_dir, 'base_full_1152.npy'))
    ft_1152 = np.load(os.path.join(activations_dir, 'ft_full_1152.npy'))

print(f"Base 1152-d shape: {base_1152.shape}")
print(f"FT 1152-d shape:   {ft_1152.shape}")

def ablate_and_evaluate(model, acts_1152, labels, top_k_neurons, dataset_mean_acts):
    """
    Proxy inference ablation: splits the 1152-d vector, clamps top-k Text neurons, 
    passes through final classification heads, and returns per-class accuracy.
    """
    text_cls = acts_1152[:, 0:1024].copy()
    faces = acts_1152[:, 1024:1088]
    specs = acts_1152[:, 1088:1152]
    
    for n in top_k_neurons:
        text_cls[:, n] = dataset_mean_acts[n]
        
    with torch.no_grad():
        t_t = torch.tensor(text_cls, dtype=torch.float32).to(device)
        f_t = torch.tensor(faces, dtype=torch.float32).to(device)
        s_t = torch.tensor(specs, dtype=torch.float32).to(device)
        
        t_logits = model.t_out(t_t)
        v_logits = model.v_out(f_t)
        a_logits = model.a_out(s_t)
        
        all_logits = torch.stack([t_logits, v_logits, a_logits], dim=-1)
        final_logits = model.weighted_fusion(all_logits).squeeze(-1)
        preds = torch.argmax(final_logits, dim=1).cpu().numpy()
    
    accs = []
    for c in range(6):
        mask = (labels == c)
        acc = np.mean(preds[mask] == labels[mask]) if np.sum(mask) > 0 else 0.0
        accs.append(acc)
        
    return np.array(accs)

print("Ablation proxy inference scaffold ready.")

## 8. Execute Causal Selectivity Sweep & Evaluation

We now compute the baseline accuracy and evaluate the accuracy drop for ablating the top k in {1, 3, 5, 10} neurons.

According to ADR 0001, a feature set is causally class-selective if:
`Target Class Accuracy Drop` >= 2.5x `Mean Absolute Non-Target Class Drop`.

**Expected output:** Tables reporting the baseline accuracy, the ablation accuracy drops for k=5 (primary reporting target), and the final causal selectivity pass/fail per class, cleanly formatted. Artifacts (CSV and JSON) are saved to results.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from utils import set_deterministic_seed

set_deterministic_seed(seed=0)
print('=== Phase C: Causal Ablation Sweep ===')


def run_sweep(model, acts_1152, acts_tier1, labels, top_neurons, model_name):
  dataset_mean_acts = np.mean(acts_tier1[0:432], axis=0)

  # Baseline accuracy (k=0)
  baseline_accs = ablate_and_evaluate(
      model, acts_1152, labels, [], dataset_mean_acts
  )

  results = []
  for c in range(6):
    k = 5
    top_k = top_neurons[c][:k]
    ablated_accs = ablate_and_evaluate(
        model, acts_1152, labels, top_k, dataset_mean_acts
    )

    acc_drops = baseline_accs - ablated_accs
    target_drop = acc_drops[c]
    non_target_drops = np.delete(acc_drops, c)
    mean_abs_non_target_drop = np.mean(np.abs(non_target_drops))

    # ADR 0001 Causal Selectivity Threshold
    is_selective = (target_drop > 0) and (
        target_drop >= (2.5 * mean_abs_non_target_drop)
    )

    if mean_abs_non_target_drop > 0:
      ratio_val = target_drop / mean_abs_non_target_drop
      ratio_str = f'{ratio_val:.2f}'
    else:
      ratio_str = 'inf' if target_drop > 0 else '0.00'

    results.append({
        'model': model_name,
        'class': EMOTION_CLASSES[c],
        'baseline_acc': round(baseline_accs[c] * 100, 2),
        'target_drop': round(target_drop * 100, 2),
        'mean_nt_drop': round(mean_abs_non_target_drop * 100, 2),
        'ratio': ratio_str,
        'causally_selective': is_selective,
    })

  df = pd.DataFrame(results)
  print(f'\n{model_name.upper()} MODEL — k=5 Causal Ablation Results:')
  print(df.to_string(index=False))

  # Save individual CSV artifact
  csv_out_path = os.path.join(
      results_dir, f'phase_c_{model_name}_ablation_k5.csv'
  )
  df.to_csv(csv_out_path, index=False)

  # Save JSON sweep artifact for curves
  sweep_results = {}
  for k_val in K_VALUES:
    sweep_results[k_val] = {}
    for c in range(6):
      top_k = top_neurons[c][:k_val]
      ablated_accs = ablate_and_evaluate(
          model, acts_1152, labels, top_k, dataset_mean_acts
      )
      sweep_results[k_val][EMOTION_CLASSES[c]] = (
          baseline_accs - ablated_accs
      ).tolist()

  json_out_path = os.path.join(
      results_dir, f'phase_c_{model_name}_ablation_sweep.json'
  )
  with open(json_out_path, 'w') as f:
    json.dump(sweep_results, f, indent=2)

  return df, sweep_results


# Run sweeps for both models
base_df, base_sweep = run_sweep(
    base_model,
    base_1152,
    base_acts_tier1,
    labels_tier1,
    base_top_neurons,
    'base',
)
ft_df, ft_sweep = run_sweep(
    ft_model, ft_1152, ft_acts_tier1, labels_tier1, ft_top_neurons, 'finetuned'
)

# Export combined summary CSV artifact (Issue B.7)
combined_df = pd.concat([base_df, ft_df], ignore_index=True)
combined_csv_path = os.path.join(
    results_dir, 'phase_c_k5_ablation_results.csv'
)
combined_df.to_csv(combined_csv_path, index=False)
print(f'\n[SAVED] Combined Phase C Summary CSV: {combined_csv_path}')

## 8.5 Day 13 Cosine Similarity Analysis (Issue A.3 & Protocol Day 13 Fix)

Compute the per-class selectivity vector `selectivity[d] = mean(act[d] | c) - mean(act[d] | ~c)` for both base and fine-tuned models, and calculate the representational cosine similarity across all 6 emotion classes.

**Expected output:** A 6-row summary table and saved CSV artifact `results/phase_d_cosine_similarity.csv`.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from utils import set_deterministic_seed

set_deterministic_seed(seed=0)
print('=== Section 8.5: Day 13 Cosine Similarity Analysis ===')

EMOTION_CLASSES = [
    'anger',
    'disgust',
    'fear',
    'happiness',
    'sadness',
    'surprise',
]


def compute_selectivity_vector(acts, labels, class_idx):
  target_mean = np.mean(acts[labels == class_idx], axis=0)
  off_target_mean = np.mean(acts[labels != class_idx], axis=0)
  return target_mean - off_target_mean


cosine_results = []
for c in range(6):
  class_name = EMOTION_CLASSES[c]

  base_sel = compute_selectivity_vector(base_acts_tier1, labels_tier1, c)
  ft_sel = compute_selectivity_vector(ft_acts_tier1, labels_tier1, c)

  # Cosine similarity = 1 - cosine distance
  sim = 1.0 - cosine(base_sel, ft_sel)

  cosine_results.append({
      'class': class_name,
      'cosine_similarity': round(sim, 4),
      'interpretation': (
          'High Sharpening' if sim >= 0.70 else 'Representation Shift/Rotate'
      ),
  })

df_cosine = pd.DataFrame(cosine_results)
print('\nBase vs Fine-Tuned Selectivity Vector Cosine Similarity:')
print(df_cosine.to_string(index=False))

out_cos_path = os.path.join(results_dir, 'phase_d_cosine_similarity.csv')
df_cosine.to_csv(out_cos_path, index=False)
print(f'\n[SAVED] {out_cos_path}')

# Phase D — Transfer Retention Analysis

Evaluate the base model's top-5 causally selective neurons inside the fine-tuned model to compute the Transfer Retention Ratio (R), following the methodology locked in [ADR 0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Classify the fine-tuning outcome as Substrate Preservation (R >= 0.70), Substrate Reassignment (R < 0.30 with FT sparse drop), or Substrate Dispersion (R < 0.30 with FT dense drop).

## 9. Compute Transfer Retention Ratio (R)

Filter for emotion classes that proved causally selective in the base model (from Phase C) and evaluate those exact top-5 base neurons inside the fine-tuned model. The ratio of the fine-tuned accuracy drop to the base accuracy drop gives $R$, which determines the Substrate Outcome taxonomy.

In [ ]:
import os
import numpy as np
import pandas as pd
from utils import set_deterministic_seed

set_deterministic_seed(seed=0)
print('=== Phase D: Transfer Retention Analysis (All 6 Classes) ===')

# Precompute FT model baseline accuracy (k=0 ablation)
ft_dataset_mean_acts = np.mean(ft_acts_tier1[0:432], axis=0)
ft_baseline_accs = ablate_and_evaluate(
    ft_model, ft_1152, labels_tier1, [], ft_dataset_mean_acts
)

# Load Phase C results for reference
base_csv_path = os.path.join(results_dir, 'phase_c_base_ablation_k5.csv')
ft_csv_path = os.path.join(results_dir, 'phase_c_finetuned_ablation_k5.csv')
base_results_df = pd.read_csv(base_csv_path)
ft_results_df = pd.read_csv(ft_csv_path)

phase_d_results = []

# Evaluate base model top-5 neurons inside fine-tuned model for ALL 6 classes (Issue B.2)
for c in range(6):
  class_name = EMOTION_CLASSES[c]

  base_row = base_results_df[base_results_df['class'] == class_name].iloc[0]
  base_drop = base_row['target_drop']
  base_top_5 = base_top_neurons[c][:5]

  # Evaluate base neurons inside fine-tuned model
  ft_ablated_accs = ablate_and_evaluate(
      ft_model, ft_1152, labels_tier1, base_top_5, ft_dataset_mean_acts
  )
  ft_drop = round((ft_baseline_accs[c] - ft_ablated_accs[c]) * 100, 2)

  # Compute Transfer Retention Ratio R
  R = round(ft_drop / base_drop, 2) if base_drop > 0 else 0.00

  ft_is_selective = ft_results_df.loc[
      ft_results_df['class'] == class_name, 'causally_selective'
  ].values[0]

  # Outcome Taxonomy (ADR 0001)
  if R >= 0.70:
    outcome = 'Substrate Preservation'
  elif R < 0.30:
    if ft_is_selective:
      outcome = 'Substrate Reassignment'
    else:
      outcome = 'Substrate Dispersion'
  else:
    outcome = 'Indeterminate (Partial Retention)'

  phase_d_results.append({
      'class': class_name,
      'base_top5_neurons': str(base_top_5.tolist()),
      'base_target_drop': base_drop,
      'ft_target_drop': ft_drop,
      'retention_ratio_R': R,
      'ft_is_selective': ft_is_selective,
      'substrate_outcome': outcome,
  })

df_d = pd.DataFrame(phase_d_results)
print('\nFT MODEL — Evaluation of Base Top-5 Neurons Across All Classes:')
print(df_d.to_string(index=False))

out_path_d = os.path.join(results_dir, 'phase_d_transfer_retention.csv')
df_d.to_csv(out_path_d, index=False)
print(f'\n[SAVED] {out_path_d}')